<a href="https://colab.research.google.com/github/ashesh-0/GoogleColabRepos/blob/main/AlphaFold2_fulllength_kaggle.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#ColabFold v1.5.5: AlphaFold2 w/ MMseqs2 BATCH

<img src="https://raw.githubusercontent.com/sokrypton/ColabFold/main/.github/ColabFold_Marv_Logo_Small.png" height="256" align="right" style="height:256px">

Easy to use AlphaFold2 protein structure [(Jumper et al. 2021)](https://www.nature.com/articles/s41586-021-03819-2) and complex [(Evans et al. 2021)](https://www.biorxiv.org/content/10.1101/2021.10.04.463034v1) prediction using multiple sequence alignments generated through MMseqs2. For details, refer to our manuscript:

[Mirdita M, Schütze K, Moriwaki Y, Heo L, Ovchinnikov S, Steinegger M. ColabFold: Making protein folding accessible to all.
*Nature Methods*, 2022](https://www.nature.com/articles/s41592-022-01488-1)

**Usage**

`input_dir` directory with only fasta files or MSAs stored in Google Drive. MSAs need to be A3M formatted and have an `.a3m` extention. For MSAs MMseqs2 will not be called.

`result_dir` results will be written to the result directory in Google Drive

Old versions: [v1.4](https://colab.research.google.com/github/sokrypton/ColabFold/blob/v1.4.0/batch/AlphaFold2_batch.ipynb), [v1.5.1](https://colab.research.google.com/github/sokrypton/ColabFold/blob/v1.5.1/batch/AlphaFold2_batch.ipynb), [v1.5.2](https://colab.research.google.com/github/sokrypton/ColabFold/blob/v1.5.2/batch/AlphaFold2_batch.ipynb), [v1.5.3-patch](https://colab.research.google.com/github/sokrypton/ColabFold/blob/56c72044c7d51a311ca99b953a71e552fdc042e1/batch/AlphaFold2_batch.ipynb)

<strong>For more details, see <a href="#Instructions">bottom</a> of the notebook and checkout the [ColabFold GitHub](https://github.com/sokrypton/ColabFold). </strong>

-----------

### News
- <b><font color='green'>2023/07/31: The ColabFold MSA server is back to normal. It was using older DB (UniRef30 2202/PDB70 220313) from 27th ~8:30 AM CEST to 31st ~11:10 AM CEST.</font></b>
- <b><font color='green'>2023/06/12: New databases! UniRef30 updated to 2023_02 and PDB to 230517. We now use PDB100 instead of PDB70 (see notes in the [main](https://colabfold.com) notebook).</font></b>
- <b><font color='green'>2023/06/12: We introduced a new default pairing strategy: Previously, for multimer predictions with more than 2 chains, we only pair if all sequences taxonomically match ("complete" pairing). The new default "greedy" strategy pairs any taxonomically matching subsets.</font></b>

In [14]:
! ls

In [1]:
#@title Mount google drive
# from google.colab import drive
# drive.mount('/content/drive')
from sys import version_info
python_version = f"{version_info.major}.{version_info.minor}"

In [2]:
#@title Input protein sequence, then hit `Runtime` -> `Run all`
import os
amyloid_status = "non_amyloid" #@param ["amyloid", "non_amyloid"]
fold_k = 'a' #@param ["a", "b", "c", "d", "e", "f"]
input_dir = os.path.join('/kaggle/input/datasets/silence2/full-length-light-chains/alphafold_inputs/',amyloid_status,fold_k)
result_dir = os.path.join('/kaggle/working/output', amyloid_status)
gdrive_output_folder_id = "1tVIGDMqryYyfFipmWoy377yDjSSk5MW0"#@param {type: "string"}
# amyloid_status = "amyloid" #@param ["amyloid", "non_amyloid"]
# sequence_type = "globally_randomized_full_length_no_leader_peptide" #@param ["full_length", "just_vl_domain", "full_length_no_leader_peptide","randomized_full_length_no_leader_peptide", "globally_randomized_full_length_no_leader_peptide"]
# input_dir = os.path.join('//home/ashesh/Documents/data/ALAmyloidosis_fulllength/full_length_fasta_data', sequence_type,amyloid_status)
# root_result_dir = '/home/ashesh/Documents/data/ALAmyloidosis_fulllength/structured_colabfold_outputs' #@param {type:"string"}
# result_dir = os.path.join(root_result_dir, sequence_type,amyloid_status)

# number of models to use
#@markdown ---
#@markdown ### Advanced settings
msa_mode = "MMseqs2 (UniRef+Environmental)" #@param ["MMseqs2 (UniRef+Environmental)", "MMseqs2 (UniRef only)","single_sequence","custom"]
num_models = 5 #@param [1,2,3,4,5] {type:"raw"}
num_recycles = 3 #@param [1,3,6,12,24,48] {type:"raw"}
stop_at_score = 100 #@param {type:"string"}
#@markdown - early stop computing models once score > threshold (avg. plddt for "structures" and ptmscore for "complexes")
use_custom_msa = False
num_relax = 0 #@param [0, 1, 5] {type:"raw"}
use_amber = num_relax > 0
relax_max_iterations = 200 #@param [0,200,2000] {type:"raw"}
use_templates = False #@param {type:"boolean"}
do_not_overwrite_results = True #@param {type:"boolean"}
zip_results = False #@param {type:"boolean"}


In [3]:
assert os.path.exists(input_dir)

In [4]:
# # skipping those which are already done.
# from datetime import datetime
# import os
# import shutil

# input_dir=f"/content/remaining_inputs_{datetime.now().strftime('%Y%m%d_%H%M')}"
# os.makedirs(input_dir, exist_ok=False)

# for fname in os.listdir(raw_input_dir):
#   if fname.endswith('.fasta'):
#     completed_fname = fname.replace('.fasta','')+ '.done.txt'
#     if os.path.exists(os.path.join(result_dir, completed_fname)):
#       print(f'Ignoring {fname} since it is done in previous runs')
#     # copy the file to new output
#     shutil.copy(os.path.join(raw_input_dir, fname), os.path.join(input_dir, fname))
#   else:
#     print(f'Ignoring {fname}')

In [5]:
#@title Install dependencies
%%bash -s $use_amber $use_templates $python_version

set -e

USE_AMBER=$1
USE_TEMPLATES=$2
PYTHON_VERSION=$3

if [ ! -f COLABFOLD_READY ]; then
  # install dependencies
  # We have to use "--no-warn-conflicts" because colab already has a lot preinstalled with requirements different to ours
  pip install -q --no-warn-conflicts "colabfold[alphafold-minus-jax] @ git+https://github.com/sokrypton/ColabFold"
  if [ -n "${TPU_NAME}" ]; then
    pip install -q --no-warn-conflicts -U dm-haiku==0.0.10 jax==0.3.25
  fi
  ln -s /usr/local/lib/python3.*/dist-packages/colabfold colabfold
  ln -s /usr/local/lib/python3.*/dist-packages/alphafold alphafold
  # hack to fix TF crash
  rm -f /usr/local/lib/python3.*/dist-packages/tensorflow/core/kernels/libtfkernel_sobol_op.so
  touch COLABFOLD_READY
fi

# Download params (~1min)
python -m colabfold.download

# setup conda
if [ ${USE_AMBER} == "True" ] || [ ${USE_TEMPLATES} == "True" ]; then
  if [ ! -f CONDA_READY ]; then
    wget -qnc https://github.com/conda-forge/miniforge/releases/download/25.3.1-0/Miniforge3-25.3.1-0-Linux-x86_64.sh
    bash Miniforge3-25.3.1-0-Linux-x86_64.sh -bfp /usr/local 2>&1 1>/dev/null
    rm Miniforge3-25.3.1-0-Linux-x86_64.sh
    conda config --set auto_update_conda false
    touch CONDA_READY
  fi
fi
# setup template search
if [ ${USE_TEMPLATES} == "True" ] && [ ! -f HH_READY ]; then
  conda install -y -q -c conda-forge -c bioconda kalign2=2.04 hhsuite=3.3.0 python="${PYTHON_VERSION}" 2>&1 1>/dev/null
  touch HH_READY
fi
# setup openmm for amber refinement
if [ ${USE_AMBER} == "True" ] && [ ! -f AMBER_READY ]; then
  conda install -y -q -c conda-forge openmm=8.2.0 python="${PYTHON_VERSION}" pdbfixer 2>&1 1>/dev/null
  touch AMBER_READY
fi

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 248.4/248.4 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 50.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 374.3/374.3 kB 19.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 274.0/274.0 MB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 81.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 46.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 324.3/324.3 kB 16.3 MB/s eta 0:00:00


  file.extractall(path=params_dir)


In [6]:
!pip install google-api-python-client google-auth-oauthlib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 323.4/323.4 kB 6.8 MB/s eta 0:00:00
  Attempting uninstall: protobuf
    Found existing installation: protobuf 7.34.1
    Uninstalling protobuf-7.34.1:
      Successfully uninstalled protobuf-7.34.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.35.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.25.1 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
google-ai-generativelanguage 0.6.15 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<6.0.0dev,>=3.20.2, but you have protobuf 6.33.6 which is incompatible.
tensorflow 2.19.0 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<6.0.0dev,>=3.20.3, but you have protobuf 6.33.6 which is incompatible.
grpcio-status 1.71.2 requires protobuf<6.0dev,>=5.26.1, b

In [8]:

from google.auth.transport.requests import Request
from googleapiclient.discovery import build
from googleapiclient.http import MediaFileUpload
import pickle
def gdrive_service():
  # Load credentials
  with open('/kaggle/input/datasets/silence2/gdrive-pickle/token.pickle', 'rb') as f:
      creds = pickle.load(f)

  # Auto-refresh token if expired
  if creds.expired and creds.refresh_token:
      creds.refresh(Request())
      with open('/kaggle/input/datasets/silence2/gdrive-pickle/token.pickle', 'wb') as f:
          pickle.dump(creds, f)

  service = build('drive', 'v3', credentials=creds)
  return service

def upload_to_drive(fpath, service, parent_id):
  media = MediaFileUpload(fpath, resumable=True)
  body = {'name': os.path.basename(fpath), 'parents': [parent_id]}
  f = service.files().create(body=body, media_body=media, fields='id').execute()
  print(f"Uploaded successfully! File ID: {f.get('id')}")


def colabfold_done_file(filename: str) -> bool:
    # ".done.txt"
    return filename.endswith('.done.txt')

def download_files_from_gdrive(folder_id: str, desirability_criteria, output_dir: str, service):
    """
    Downloads all files with a specific extension from a given Google Drive folder.

    Args:
        folder_id (str): The ID of the Google Drive folder.
        extension (str): The file extension to filter by (e.g., '.csv', '.nii.gz').
        output_dir (str): The local directory to save the downloaded files.
        credentials_path (str): Path to the Google Drive API credentials.json file.
    """
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)


    # Query to list files in the specific folder that are not trashed
    query = f"'{folder_id}' in parents and trashed=false"

    page_token = None
    while True:
        response = service.files().list(q=query,
                                        spaces='drive',
                                        fields='nextPageToken, files(id, name)',
                                        pageToken=page_token).execute()

        for file in response.get('files', []):
            file_name = file.get('name')
            # Check if the file ends with the desired extension
            if file_name and desirability_criteria(file_name):
                file_path = os.path.join(output_dir, file_name)
                if os.path.exists(file_path):
                    print(f"File {file_name} already exists. Skipping download.")
                    continue
                file_id = file.get('id')

                print(f"Downloading {file_name} (ID: {file_id})...")
                request = service.files().get_media(fileId=file_id)

                with io.FileIO(file_path, 'wb') as fh:
                    downloader = MediaIoBaseDownload(fh, request)
                    done = False
                    while done is False:
                        status, done = downloader.next_chunk()
                        if status:
                            print(f"Download {int(status.progress() * 100)}%.")

                print(f"Saved to {file_path}")

        # Check if there are more files to fetch
        page_token = response.get('nextPageToken', None)
        if page_token is None:
            break



In [10]:
# download whatever has been completed.
service = gdrive_service()
download_files_from_gdrive(gdrive_output_folder_id, colabfold_done_file, result_dir, service)

In [12]:
result_dir, input_dir

('/kaggle/working/output/non_amyloid',
 '/kaggle/input/datasets/silence2/full-length-light-chains/alphafold_inputs/non_amyloid/a')

In [13]:
#@title Run Prediction

import sys

from colabfold.batch import get_queries, run
from colabfold.download import default_data_dir
from colabfold.utils import setup_logging
from pathlib import Path

# For some reason we need that to get pdbfixer to import
if use_amber and f"/usr/local/lib/python{python_version}/site-packages/" not in sys.path:
    sys.path.insert(0, f"/usr/local/lib/python{python_version}/site-packages/")

setup_logging(Path(result_dir).joinpath("log.txt"))

queries, is_complex = get_queries(input_dir)
batch_count = 2
for i in range(0, len(queries), batch_count):
  run(
      queries=queries[batch_count*i: batch_count*i + batch_count],
      result_dir=result_dir,
      use_templates=use_templates,
      num_relax=num_relax,
      relax_max_iterations=relax_max_iterations,
      msa_mode=msa_mode,
      model_type="auto",
      num_models=num_models,
      num_recycles=num_recycles,
      model_order=[1, 2, 3, 4, 5],
      is_complex=is_complex,
      data_dir=default_data_dir,
      keep_existing_results=do_not_overwrite_results,
      rank_by="auto",
      pair_mode="unpaired+paired",
      stop_at_score=stop_at_score,
      zip_results=zip_results,
      user_agent="colabfold/google-colab-batch",
  )
  service = gdrive_service()
  for fname in os.listdir(result_dir):
    upload_to_drive(os.path.join(result_dir,fname), service, gdrive_output_folder_id)
  print('Uploaded')
  print('')

  # remove all files in that directory.
  import shutil
  import os

  # Delete the entire directory tree
  shutil.rmtree(result_dir)

  # Recreate the empty directory
  os.makedirs(result_dir)
  break

2026-05-13 12:58:48,006 WARNING: no GPU detected, will be using CPU


KeyboardInterrupt: 

/usr/lib/python3.12/pty.py:95: RuntimeWarning: os.fork() was called. os.fork() is incompatible with multithreaded code, and JAX is multithreaded, so this will likely lead to a deadlock.
  pid, fd = os.forkpty()


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 323.4/323.4 kB 12.0 MB/s eta 0:00:00
  Attempting uninstall: protobuf
    Found existing installation: protobuf 7.34.1
    Uninstalling protobuf-7.34.1:
      Successfully uninstalled protobuf-7.34.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.35.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.25.1 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
google-ai-generativelanguage 0.6.15 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<6.0.0dev,>=3.20.2, but you have protobuf 6.33.6 which is incompatible.
tensorflow 2.19.0 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<6.0.0dev,>=3.20.3, but you have protobuf 6.33.6 which is incompatible.
grpcio-status 1.71.2 requires protobuf<6.0dev,>=5.26.1, 

In [12]:
# from google.auth.transport.requests import Request
# from googleapiclient.discovery import build
# from googleapiclient.http import MediaFileUpload
# import pickle
# def gdrive_service():
#   # Load credentials
#   with open('/kaggle/input/datasets/silence2/gdrive-pickle/token.pickle', 'rb') as f:
#       creds = pickle.load(f)

#   # Auto-refresh token if expired
#   if creds.expired and creds.refresh_token:
#       creds.refresh(Request())
#       with open('/kaggle/input/datasets/silence2/gdrive-pickle/token.pickle', 'wb') as f:
#           pickle.dump(creds, f)

#   service = build('drive', 'v3', credentials=creds)
#   return service


# Upload
media = MediaFileUpload('/kaggle/working/ashesh.txt', resumable=True)
body = {'name': 'ashesh.txt', 'parents': ['18MTZj9eUowgBeX8jfDQflC1P52OfH_0S']}
f = service.files().create(body=body, media_body=media, fields='id').execute()
print(f"Uploaded successfully! File ID: {f.get('id')}")

Uploaded successfully! File ID: 1G9uXC8gVHHK7xHpAgwgj99sY7OoVQ_56
